# ASCA AI - Guardrail & Llama 3.1 8B (OpenRouter) Integration Notebook

This notebook tests the **Pydantic Guardrail Engine** (`src/agents/guardrail.py`) connected to **Meta Llama 3.1 8B via OpenRouter** (`meta-llama/llama-3.1-8b-instruct`).
It validates zero hallucinations, strict schema enforcement, and auto-retry logic.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.agents.guardrail import (
    MarketInsight,
    B2BMatchRecommendation,
    ExecutiveAdvisoryBlueprint,
    validate_blueprint_output,
    generate_validated_blueprint_with_openrouter,
    VALID_CROPS,
    VALID_CENTERS
)
from src.infrastructure.config import config

print("Guardrail & Llama 3.1 8B integration modules loaded successfully!")
print(f"Valid Economic Centers: {list(VALID_CENTERS)}")
print(f"Supported Sri Lankan Crops ({len(VALID_CROPS)}): {sorted(list(VALID_CROPS))}")
print(f"Default OpenRouter Model: {config.models.get('llm_providers', {}).get('openrouter', {}).get('default_model')}")
print(f"OpenRouter API Key Configured: {'YES' if config.env.OPENROUTER_API_KEY and not config.env.OPENROUTER_API_KEY.startswith('your_') else 'NO (Set in .env file)'}")

Guardrail & Llama 3.1 8B integration modules loaded successfully!
Valid Economic Centers: ['DAMBULLA', 'THAMBUTHTHEGAMA']
Supported Sri Lankan Crops (20): ['banana', 'beans', 'beetroot', 'bitter_gourd', 'brinjal', 'cabbage', 'capsicum', 'carrot', 'cucumber', 'eggplant', 'green_chilli', 'leeks', 'lime', 'mango', 'papaya', 'passion_fruit', 'pineapple', 'pumpkin', 'snake_gourd', 'tomato']
Default OpenRouter Model: meta-llama/llama-3.1-8b-instruct
OpenRouter API Key Configured: YES


## Section 1: Offline Guardrail Schema Validation Test

In [3]:
sample_llm_json = '''
{
    "title": "Dambulla Tomato Surplus & Price Drop Advisory",
    "target_centers": ["DAMBULLA"],
    "primary_surplus_crops": ["tomato"],
    "market_insights": [
        {
            "center_id": "DAMBULLA",
            "crop_name": "tomato",
            "current_wholesale_price_lkr": 200.0,
            "predicted_wholesale_price_lkr": 120.0,
            "supply_volume_tons": 50.0,
            "surplus_anomaly_detected": true,
            "risk_level": "HIGH"
        }
    ],
    "b2b_recommendations": [
        {
            "buyer_code": "BUYER_SAUCE_01",
            "company_name": "Lanka Processing Mills",
            "crop_name": "tomato",
            "matched_volume_tons": 25.0,
            "fefo_risk_score": 0.2,
            "recommended_action": "Negotiate 25 tons allocation to Lanka Processing Mills."
        }
    ],
    "executive_summary_sinhala": "දඹුල්ලේ තක්කාලි අස්වැන්න වැඩිවී මිල 40%කින් අඩුවීමේ අවදානමක් ඇත. ටොන් 25ක් සෝස් නිෂ්පාදකයින්ට යොමු කෙරේ.",
    "telegram_alert_text": "⚠️ දඹුල්ල අතිරික්ත අවදානම: තක්කාලි ටොන් 25ක් සෝස් කර්මාන්තශාලා වෙත auto-match විය.",
    "confidence_score": 0.95
}
'''

res = validate_blueprint_output(sample_llm_json)
print(f"Validation Result: {res.is_valid}")
if res.is_valid:
    print("Title:", res.blueprint.title)
    print("Telegram Alert (Sinhala):", res.blueprint.telegram_alert_text)

2026-07-30 11:25:12 | INFO     | src.agents.guardrail:validate_blueprint_output:162 - Executive Advisory Blueprint successfully validated through Guardrail.
Validation Result: True
Title: Dambulla Tomato Surplus & Price Drop Advisory
Telegram Alert (Sinhala): ⚠️ දඹුල්ල අතිරික්ත අවදානම: තක්කාලි ටොන් 25ක් සෝස් කර්මාන්තශාලා වෙත auto-match විය.


## Section 2: Live Llama 3.1 8B (OpenRouter) + Pydantic Guardrail Validation Test

*Note: Ensure `OPENROUTER_API_KEY` is set in your `.env` file to execute live LLM calls.*

In [ ]:
market_scenario_prompt = '''
Market Scouting Report for Dambulla Economic Center:
- Date: Today
- Location: DAMBULLA
- Crop: Tomato
- Current Wholesale Price: 250 LKR/kg
- Forecasted Price in 14 days: 130 LKR/kg (48% expected price crash!)
- Estimated Excess Harvest: 60 Metric Tons
- Matched Buyer: BUYER_SAUCE_DAMBULLA (Lanka Sauce Factory, Capacity: 30 Tons/day)

Generate an Executive Advisory Blueprint in strict JSON format.
Include executive_summary_sinhala and telegram_alert_text in Sinhala.
'''

if not config.env.OPENROUTER_API_KEY or config.env.OPENROUTER_API_KEY.startswith("your_"):
    print("⚠️ OPENROUTER_API_KEY is not configured in .env file yet!")
    print("Please set OPENROUTER_API_KEY=your_actual_key inside the .env file to run live LLM validation.")
else:
    print("🚀 Invoking Meta Llama 3.1 8B via OpenRouter with Pydantic Guardrail Protection...")
    result = generate_validated_blueprint_with_openrouter(
        market_scenario_prompt,
        model_name="meta-llama/llama-3.1-8b-instruct"
    )
    
    print(f"Validation Passed: {result.is_valid}")
    if result.is_valid:
        print("\n--- VALIDATED EXECUTIVE BLUEPRINT (Llama 3.1 8B) ---")
        print("Title:", result.blueprint.title)
        print("Target Centers:", result.blueprint.target_centers)
        print("Sinhala Summary:", result.blueprint.executive_summary_sinhala)
        print("Telegram Alert Text:", result.blueprint.telegram_alert_text)
    else:
        print("Validation Errors:", result.error_messages)

🚀 Invoking Meta Llama 3.1 8B via OpenRouter with Pydantic Guardrail Protection...
2026-07-30 13:04:04 | INFO     | src.agents.guardrail:generate_validated_blueprint_with_openrouter:186 - Initiating OpenRouter LLM Call [meta-llama/llama-3.1-8b-instruct] with Pydantic Guardrail Protection...
2026-07-30 13:04:04 | INFO     | src.infrastructure.llm_loader:get_llm:16 - Loading OpenRouter LLM: meta-llama/llama-3.1-8b-instruct
2026-07-30 13:04:04 | INFO     | src.agents.guardrail:generate_validated_blueprint_with_openrouter:230 - OpenRouter [meta-llama/llama-3.1-8b-instruct] Attempt 1/3...
